In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. File paths
BASE_DIR = Path(__file__).resolve().parent

DATA_FILE = BASE_DIR / "GridEye_combined_clean_dataset.csv"
MODEL_FILE = BASE_DIR / "GridEye_retrained_random_forest.pkl"
REPORT_FILE = BASE_DIR / "GridEye_retraining_report.txt"

# 2. Expected model features, in exact order
FEATURES = [
    "V1", "I1", "P1", "PF1",
    "V2", "I2", "P2", "PF2",
    "Power_Diff", "Current_Diff", "Voltage_Diff"
]

# 3. Read combined dataset
df = pd.read_csv(DATA_FILE)

# Standardize column names
df.columns = [
    str(col).strip().replace(" ", "_")
    for col in df.columns
]

print("Dataset columns:", df.columns.tolist())
print("Original samples:", len(df))

# 4. Find the target label column
possible_labels = ["Label", "label", "Class", "class",
                   "Condition", "condition", "Target", "target"]

label_col = next(
    (col for col in possible_labels if col in df.columns),
    None
)

if label_col is None:
    raise ValueError(
        "No label column found. The dataset must identify "
        "the condition for each row. Inspect its columns first."
    )

missing = [col for col in FEATURES if col not in df.columns]

if missing:
    raise ValueError(f"Missing required features: {missing}")

# 5. Clean incomplete and invalid rows
for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=FEATURES + [label_col]).copy()

# Remove blank labels
df[label_col] = df[label_col].astype(str).str.strip()
df = df[df[label_col] != ""]

# Calculate differences only if the dataset contains no
# usable values for those columns.
difference_pairs = {
    "Power_Diff": ("P1", "P2"),
    "Current_Diff": ("I1", "I2"),
    "Voltage_Diff": ("V1", "V2")
}

for col, (a, b) in difference_pairs.items():
    if col not in df.columns:
        df[col] = df[a] - df[b]

# Confirm that all four expected conditions are represented.
expected_classes = {
    "Direct Theft (Bypass)",
    "Idle / Both OFF",
    "Normal Operation (Consumer Only)",
    "Partial Theft (Both Loads Active)"
}

print("\nClass counts:")
print(df[label_col].value_counts())

if df[label_col].nunique() != 4:
    raise ValueError(
        "Expected four classes. Check the actual label names "
        "before training."
    )

if df[label_col].value_counts().min() < 5:
    raise ValueError("Each class needs at least five valid samples.")

X = df[FEATURES].copy()
y = df[label_col].copy()

# Save cleaned combined data
df.to_csv(DATA_FILE, index=False)

# 6. Split before evaluating
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 7. Train Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 8. Evaluate on held-out test data
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

report = classification_report(
    y_test, predictions, zero_division=0
)

matrix = confusion_matrix(
    y_test, predictions, labels=model.classes_
)

# Cross-validation on the full dataset
cv_scores = cross_val_score(
    model, X, y, cv=5, scoring="accuracy", n_jobs=-1
)

# 9. Save the trained model
joblib.dump(model, MODEL_FILE)

report_text = f"""
GridEye Random Forest Training Report
=====================================
Valid samples: {len(df)}
Features: {FEATURES}
Classes: {list(model.classes_)}
Trees: {model.n_estimators}

Held-out test accuracy: {accuracy:.4f}
5-fold CV scores: {cv_scores}
Mean CV accuracy: {cv_scores.mean():.4f}
CV standard deviation: {cv_scores.std():.4f}

Classification report:
{report}

Confusion matrix:
{matrix}
"""

REPORT_FILE.write_text(report_text, encoding="utf-8")

print("\n", report_text)
print("Saved model:", MODEL_FILE)
print("Saved report:", REPORT_FILE)


NameError: name '__file__' is not defined